# Experiment: long-text handling, chunk averaging
Reference: XLM-R head+tail (same fine-tuned model). Inference only: the whole text is split into 510-token windows (last window aligned to the end, at most `MAX_CHUNKS` evenly spaced), logits are averaged.
Also saves chunk-spread features (std / max / last chunk) for the ensemble: fusion = human beginning + rewritten rest, so chunk predictions should disagree.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

LABELS = ['human', 'ai', 'fusion']
BODY = 510
MAX_CHUNKS = 8
BATCH = 64
INPUT = Path('/kaggle/input')
MODEL_DIR = next(INPUT.rglob('xlmr_headtail/config.json')).parent

train = pd.read_parquet(next(INPUT.rglob('train_clean.parquet')), columns=['id', 'text', 'lang', 'domain', 'label', 'fold'])
va = train[train.fold == 'val'].reset_index(drop=True)
test = pd.read_parquet(next(INPUT.rglob('test_clean.parquet')))
del train

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
tok.model_max_length = 10**6
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()
if device == 'cuda':
    model = model.half()
    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)
assert [model.module if hasattr(model, 'module') else model][0].config.id2label[0] == LABELS[0]
len(va), len(test), str(MODEL_DIR)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

(54537,
 25000,
 '/kaggle/input/notebooks/mariiahamaniuk/hw3-encoder-real/xlmr_headtail')

## Chunking

In [2]:
def chunk(ids):
    starts = list(range(0, max(len(ids) - BODY, 0) + 1, BODY))
    if starts[-1] + BODY < len(ids):
        starts.append(len(ids) - BODY)
    if len(starts) > MAX_CHUNKS:
        starts = [starts[int(k)] for k in np.linspace(0, len(starts) - 1, MAX_CHUNKS)]
    return [[tok.cls_token_id] + ids[s:s + BODY] + [tok.sep_token_id] for s in starts]

def make_chunks(texts):
    chunks, doc = [], []
    for i, ids in enumerate(tok(list(texts), add_special_tokens=False)['input_ids']):
        c = chunk(ids)
        chunks += c
        doc += [i] * len(c)
    return chunks, np.array(doc)

@torch.no_grad()
def predict(chunks):
    order = np.argsort([len(c) for c in chunks])
    out = np.zeros((len(chunks), len(LABELS)), np.float32)
    for b in range(0, len(order), BATCH):
        idx = order[b:b + BATCH]
        batch = tok.pad({'input_ids': [chunks[i] for i in idx]}, return_tensors='pt').to(device)
        out[idx] = model(**batch).logits.float().cpu().numpy()
    return out

def aggregate(logits, doc, n_docs):
    df = pd.DataFrame(logits, columns=LABELS).assign(doc=doc)
    probs = pd.DataFrame(torch.softmax(torch.tensor(logits), -1).numpy(), columns=LABELS).assign(doc=doc)
    g, gp = df.groupby('doc'), probs.groupby('doc')
    mean = g[LABELS].mean().reindex(range(n_docs))
    feats = pd.concat([
        gp[LABELS].std().fillna(0).add_suffix('_std'),
        gp[LABELS].max().add_suffix('_max'),
        gp[LABELS].min().add_suffix('_min'),
        gp[LABELS].last().add_suffix('_last'),
        g.size().rename('n_chunks'),
    ], axis=1).reindex(range(n_docs))
    return mean, feats

res = {}
for name, df in [('val', va), ('test', test)]:
    chunks, doc = make_chunks(df.text)
    print(name, 'chunks:', len(chunks), 'per doc:', round(len(chunks) / len(df), 2))
    res[name] = aggregate(predict(chunks), doc, len(df))

val chunks: 112170 per doc: 2.06
test chunks: 50275 per doc: 2.01


## Save

In [3]:
for name, df in [('val', va), ('test', test)]:
    mean, feats = res[name]
    mean.set_axis(df.id).rename_axis('id').to_csv(f'{name}_logits_xlmr_chunks.csv')
    feats.set_axis(df.id).rename_axis('id').to_csv(f'{name}_chunkfeats_xlmr.csv')

## Comparison with head+tail on the same val fold

In [4]:
y = va.set_index('id').label
mean, feats = res['val']
mean.index = feats.index = va.id
headtail = pd.read_csv(next(INPUT.rglob('val_logits_xlmr_headtail.csv')), index_col='id').loc[va.id]
soft = lambda x: torch.softmax(torch.tensor(x.values), -1).numpy()

variants = {
    'head+tail': headtail.values,
    'chunks mean': mean.values,
    'last chunk': feats[[f'{l}_last' for l in LABELS]].values,
    'head+tail + chunks mean': soft(headtail) + soft(mean),
}
def scores(p):
    pred = np.array(LABELS)[p.argmax(1)]
    f = f1_score(y, pred, labels=LABELS, average=None)
    return {**dict(zip(LABELS, f)), 'macro': f.mean()}

pd.DataFrame({k: scores(p) for k, p in variants.items()}).T.round(4)

,human,ai,fusion,macro
head+tail,0.8431,0.9875,0.8740,0.9015
chunks mean,0.7770,0.9870,0.8514,0.8718
last chunk,0.8339,0.9791,0.8545,0.8892
head+tail + chunks mean,0.8418,0.9883,0.8826,0.9042


In [5]:
v = va.set_index('id')
pd.DataFrame({
    k: pd.concat({by: v.assign(pred=np.array(LABELS)[p.argmax(1)]).groupby(by).apply(lambda d: f1_score(d.label, d.pred, average='macro'))
                  for by in ['lang', 'domain']})
    for k, p in variants.items()
}).round(4)

/tmp/ipykernel_50/861747616.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  k: pd.concat({by: v.assign(pred=np.array(LABELS)[p.argmax(1)]).groupby(by).apply(lambda d: f1_score(d.label, d.pred, average='macro'))
/tmp/ipykernel_50/861747616.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  k: pd.concat({by: v.assign(pred=np.array(LABELS)[p.argmax(1)]).groupby(by).apply(lambda d: f1_score(d.label, d.pred, average=

head+tail  chunks mean  last chunk  \
lang   de-DE                      0.9024       0.7979      0.8780   
       en-EN                      0.9013       0.8886      0.8918   
domain arxiv_papers               0.7684       0.6305      0.6926   
       blog_authorship_corpus     0.8246       0.8219      0.8237   
       bundestag                  0.9026       0.7981      0.8783   
       cnn_news                   0.9526       0.9605      0.9530   
       euro_court_cases           0.8331       0.8027      0.8120   
       gutenberg                  0.6958       0.6626      0.6684   
       house_of_commons           0.9379       0.8725      0.9168   
       student_essays             0.9322       0.9205      0.9293   

                               head+tail + chunks mean  
lang   de-DE                                    0.8974  
       en-EN                                    0.9059  
domain arxiv_papers                             0.6264  
       blog_authorship_corpus                   0.8255  
       bundestag                                0.8976  
       cnn_news                                 0.9632  
       euro_court_cases                         0.8363  
       gutenberg                                0.7085  
       house_of_commons                         0.9406  
       student_essays                           0.9331

In [6]:
feats.assign(label=y.values).groupby('label')[[f'{l}_std' for l in LABELS] + ['n_chunks']].median().round(3)

,human_std,ai_std,fusion_std,n_chunks
label,,,,
ai,0.000,0.0,0.000,2.0
fusion,0.000,0.0,0.000,2.0
human,0.034,0.0,0.035,2.0
